In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/28 22:10:44 WARN Utils: Your hostname, imam, resolves to a loopback address: 127.0.1.1; using 192.168.1.13 instead (on interface wlp2s0)
26/09/28 22:10:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 22:10:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 4.2.0


In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [3]:
# Extract
# Membaca ketiga sumber data ke dalam PySpark DataFrame terpisah
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_prod = spark.read.json("tugas6_produk.json")
df_rev = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# Menampilkan jumlah baris dan skema masing-masing
print(f"Data Transaksi: {df_trx.count()} baris")
df_trx.printSchema()

print(f"Data Produk: {df_prod.count()} baris")
df_prod.printSchema()

print(f"Data Ulasan: {df_rev.count()} baris")
df_rev.printSchema()

Data Transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data Produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data Ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [4]:
# Transform : Penggabungan
from pyspark.sql.functions import col

# Menggabungkan transaksi ke produk (inner join karena produk wajib valid) 
# dan ke ulasan (left join karena tidak semua transaksi punya ulasan)
df_joined = df_trx.join(df_prod, on="product_id", how="inner") \
                  .join(df_rev, on="order_id", how="left")

# Menambahkan kolom turunan total_pendapatan (unit_terjual x harga)
df_transformed = df_joined.withColumn(
    "total_pendapatan", col("unit_terjual") * col("harga")
)

In [5]:
# Transform : Penggabungan dengan Tampilkan (show)
# Menggabungkan DataFrame
df_joined = df_trx.join(df_prod, on="product_id", how="inner") \
                  .join(df_rev, on="order_id", how="left")

df_transformed = df_joined.withColumn(
    "total_pendapatan", col("unit_terjual") * col("harga")
)

# Menampilkan output untuk verifikasi hasil gabungan
print("Contoh hasil gabungan dan total pendapatan:")
df_transformed.select("order_id", "product_id", "unit_terjual", "harga", "total_pendapatan", "rating").show(5)

Contoh hasil gabungan dan total pendapatan:
+--------+----------+------------+------+----------------+------+
|order_id|product_id|unit_terjual| harga|total_pendapatan|rating|
+--------+----------+------------+------+----------------+------+
|     TX0|        21|           2| 25000|           50000|     1|
|     TX1|         8|           6|250000|         1500000|  NULL|
|     TX2|        25|           4| 25000|          100000|     2|
|     TX3|         3|           4| 75000|          300000|     5|
|     TX4|        19|           6| 75000|          450000|  NULL|
+--------+----------+------------+------+----------------+------+
only showing top 5 rows


In [6]:
# Transform : Penanganan Data Kosong dan Pengayaan
from pyspark.sql.functions import when

# Menambahkan kolom ada_ulasan bernilai True/False SEBELUM langkah na.fill
df_enriched = df_transformed.withColumn(
    "ada_ulasan", 
    when(col("rating").isNotNull(), True).otherwise(False)
)

# Mengisi nilai kosong (null) pada rating dengan angka 0
df_etl_final = df_enriched.na.fill({"rating": 0})

In [7]:
# Transform : Penanganan Data Kosong dan Pengayaan dengan Tampilkan(show)
df_enriched = df_transformed.withColumn(
    "ada_ulasan", 
    when(col("rating").isNotNull(), True).otherwise(False)
)

df_etl_final = df_enriched.na.fill({"rating": 0})

# Menampilkan output untuk verifikasi kolom baru (ada_ulasan) dan pengisian nilai 0 pada rating
print("Contoh hasil pengayaan (ada_ulasan) dan na.fill (rating 0):")
df_etl_final.select("order_id", "rating", "ada_ulasan").filter(col("rating") == 0).show(5)

Contoh hasil pengayaan (ada_ulasan) dan na.fill (rating 0):
+--------+------+----------+
|order_id|rating|ada_ulasan|
+--------+------+----------+
|     TX1|     0|     false|
|     TX4|     0|     false|
|     TX7|     0|     false|
|     TX9|     0|     false|
|    TX10|     0|     false|
+--------+------+----------+
only showing top 5 rows


In [8]:
# Load
# Menyimpan hasil akhir ke HDFS dalam format Parquet, terpartisi berdasarkan kategori
df_etl_final.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"
)

# Verifikasi struktur folder secara rekursif di HDFS
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

# Membaca kembali dari HDFS untuk membuktikan data tersimpan utuh
df_loaded = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl")
print("Jumlah baris data yang berhasil dibaca kembali:", df_loaded.count())

-rw-r--r--   3 imam supergroup          0 2026-09-28 22:11 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - imam supergroup          0 2026-09-28 22:11 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 imam supergroup      16385 2026-09-28 22:11 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-c6ffeb99-e05b-478d-abde-60e571d60f62.c000.snappy.parquet
drwxr-xr-x   - imam supergroup          0 2026-09-28 22:11 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 imam supergroup      11159 2026-09-28 22:11 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-c6ffeb99-e05b-478d-abde-60e571d60f62.c000.snappy.parquet
drwxr-xr-x   - imam supergroup          0 2026-09-28 22:11 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 imam supergroup      10419 2026-09-28 22:11 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-c6ffeb99-e05b-478d-abde-60e571d60f62.c000.snappy.parquet
drwxr-xr-x   - imam supergroup 

In [9]:
# Insight Akhir
from pyspark.sql.functions import count

# Menghitung persentase transaksi dengan ulasan per kategori produk
df_insight = df_etl_final.groupBy("kategori").agg(
    (count(when(col("ada_ulasan") == True, True)) / count("order_id") * 100).alias("persentase_ulasan")
).orderBy("persentase_ulasan")

df_insight.show()

+------------+-----------------+
|    kategori|persentase_ulasan|
+------------+-----------------+
|     Makanan|68.84328358208955|
|   Kesehatan|68.88657648283039|
|     Fashion|70.14492753623188|
|Rumah Tangga| 70.5521472392638|
|  Elektronik|  70.659407138536|
+------------+-----------------+



Interpretasi Bisnis :
Kategori produk yang sepi ulasan menunjukkan pelanggan kurang aktif setelah berbelanja.Informasi ini akan membantu tim marketing untuk membuat strategi khusus, seperti diskon atau poin reward—untuk merangsang ulasan dari pelanggan. Karena ulasan banyak dari para customer sangat ampuh membangun kepercayaan pembeli baru dan menaikkan penjualan.